# Stage 3 -- Model-Ready: Combined Full Moments

## Input
- `Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_daily_full_moments.parquet` -- already z-scored daily full moments table
- `Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_monthly_full_moments.parquet` -- already z-scored monthly full moments table

## Purpose
Merges the z-scored daily and monthly full moments tables into a single combined dataset by forward-filling monthly features to daily frequency using `merge_asof`. Both inputs are already z-standardised -- no additional standardisation is applied here. On each trading day the model sees: daily stock moments (cwmean/cwstd/cwskew/cwkurt/spread) + macro daily features, plus the most recent month-end values of monthly stock moments + macro monthly features, plus both a daily and a monthly target.

---

## Pipeline

### Step 1: Load Both Z-Scored Tables
Both the daily and monthly full moments tables are loaded and sorted by date. Shape and date ranges are reported for both.

### Step 2: Prefix Monthly Columns
All monthly feature columns (everything except `date` and `target_monthly_return`) are prefixed with `monthly_`. This is applied before the merge to prevent any column name conflicts and to make the origin of each feature unambiguous in the combined table. Conflicts between daily and monthly column names are checked and reported before prefixing. The number of renamed columns is reported.

### Step 3: Merge_asof Monthly to Daily Frequency
`pd.merge_asof(direction='backward')` on `date` is used to forward-fill monthly features onto the daily spine. Each trading day receives the most recent month-end observation on or before that date. After the merge, rows before the first valid monthly observation (where all monthly columns would be NaN) are dropped. The number of trimmed rows and the resulting date range are reported.

### Step 4: Validate
- **No duplicate dates**
- **Zero NaN in features:** all feature columns checked; any remaining NaN listed
- **Both targets checked:** daily target NaN count and monthly target NaN count (monthly target is repeated across all trading days within a month via the forward-fill)
- **Daily and monthly target statistics:** mean and std for both
- **Forward-fill verification:** for a sample mid-month trading day, the most recent month-end date from the monthly table is identified and printed, confirming the merge_asof correctly matched the right monthly observation
- **Column breakdown:** daily features, monthly features (prefixed), daily target, monthly target, date
- **Z-score sanity check:** for one sample daily base factor and one sample monthly base factor, all five moment suffixes are shown with their mean and std (expect mean ≈ 0, std ≈ 1)

### Step 5: Save
Sorted by date and saved to parquet.

---

## Key Design Decisions
- **No z-scoring applied here.** Both inputs are already z-standardised from their respective Stage 3 notebooks. This notebook is purely a merge operation.
- **`monthly_` prefix applied before the merge** to avoid any collision between daily and monthly columns that share a base name. This is the same prefix used by the combined means notebook and the feature naming notebooks.
- **`merge_asof(direction='backward')`** carries each month-end observation forward to all trading days until the next month-end arrives. The monthly target (`target_monthly_return`) is also carried forward this way -- on each trading day within month M, the target is the return for month M+1, which is the correct next-month return.
- **Rows before first monthly observation dropped** via `dropna(subset=monthly_cols)`. This removes trading days in the daily table that precede the start of the monthly table, which would otherwise have NaN for all monthly features.

## Stage 3 Summary
This notebook completes Stage 3. The six model-ready tables are:

**means_only/**

| File | Rows | Columns |
|---|---|---|
| `model_market_daily_means.parquet` | 4,299 | ~400 |
| `model_market_monthly_means.parquet` | 205 | ~326 |
| `model_market_combined_means.parquet` | 4,299 | ~725 |

**full_moments/**

| File | Rows | Columns |
|---|---|---|
| `model_market_daily_full_moments.parquet` | 4,299 | ~1,147 |
| `model_market_monthly_full_moments.parquet` | 205 | ~1,069 |
| `model_market_combined_full_moments.parquet` | 4,299 | ~2,200+ |

All tables: zero NaN, verified targets, expanding-window z-scored, no look-ahead bias, ready for model development.

## Output
`Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_combined_full_moments.parquet` -- keyed on `date` (trading day), containing z-scored daily features, z-scored monthly features (forward-filled via merge_asof, prefixed `monthly_`), `target_daily_return`, and `target_monthly_return`

In [1]:
# %% [markdown]
# # Stage 3 — Model-Ready: Combined Full Moments
#
# Merges the z-scored daily full moments with z-scored monthly full moments
# by forward-filling monthly features to daily frequency using merge_asof.
#
# On each trading day, the model sees:
#   - Daily features: z-scored stock moments (cwmean/std/skew/kurt/spread) + macro daily
#   - Monthly features: z-scored stock moments + macro monthly (most recent month-end)
#   - Daily target: next trading day's cap-weighted return
#   - Monthly target: next month's cap-weighted return
#
# Both inputs are ALREADY z-scored — no additional standardisation needed.
#
# Input:
#   Stage_3_Model_Ready/model_market_daily_full_moments.parquet
#   Stage_3_Model_Ready/model_market_monthly_full_moments.parquet
#
# Output:
#   Stage_3_Model_Ready/model_market_combined_full_moments.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')

DAILY_PATH = BASE_DIR / 'model_market_daily_full_moments.parquet'
MONTHLY_PATH = BASE_DIR / 'model_market_monthly_full_moments.parquet'

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD BOTH Z-SCORED TABLES
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD BOTH Z-SCORED TABLES")
print("=" * 90)

daily = pd.read_parquet(DAILY_PATH)
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values('date').reset_index(drop=True)

print(f"\n  Daily:   {daily.shape[0]:,} rows × {daily.shape[1]} columns")
print(f"           {daily['date'].min().date()} → {daily['date'].max().date()}")

monthly = pd.read_parquet(MONTHLY_PATH)
monthly['date'] = pd.to_datetime(monthly['date'])
monthly = monthly.sort_values('date').reset_index(drop=True)

print(f"  Monthly: {monthly.shape[0]} rows × {monthly.shape[1]} columns")
print(f"           {monthly['date'].min().date()} → {monthly['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: PREFIX MONTHLY COLUMNS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: PREFIX MONTHLY COLUMNS")
print("=" * 90)

monthly_meta = ['date', 'target_monthly_return']
monthly_features = [c for c in monthly.columns if c not in monthly_meta]

# Check for conflicts
daily_cols = set(daily.columns) - {'date'}
monthly_feature_set = set(monthly_features)
conflicts = daily_cols & monthly_feature_set

if conflicts:
    print(f"\n  Column conflicts found ({len(conflicts)}):")
    for c in sorted(list(conflicts))[:10]:
        print(f"    {c}")
    if len(conflicts) > 10:
        print(f"    ... and {len(conflicts) - 10} more")
else:
    print(f"\n  No column name conflicts")

# Prefix all monthly features
rename_map = {c: f'monthly_{c}' for c in monthly_features}
monthly = monthly.rename(columns=rename_map)

print(f"\n  Prefixed {len(rename_map)} monthly feature columns with 'monthly_'")
print(f"  Monthly columns after rename: {monthly.shape[1]}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: MERGE_ASOF MONTHLY TO DAILY FREQUENCY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: MERGE_ASOF MONTHLY TO DAILY FREQUENCY")
print("=" * 90)

daily = daily.sort_values('date')
monthly = monthly.sort_values('date')

combined = pd.merge_asof(
    daily,
    monthly,
    on='date',
    direction='backward'
)

print(f"\n  After merge_asof: {combined.shape[0]:,} rows × {combined.shape[1]} columns")

# Identify monthly columns
monthly_cols = [c for c in combined.columns if c.startswith('monthly_')] + ['target_monthly_return']

# Drop rows before first monthly observation
pre_trim = len(combined)
combined = combined.dropna(subset=monthly_cols).reset_index(drop=True)
trimmed = pre_trim - len(combined)

print(f"  Trimmed {trimmed} rows before first monthly observation")
print(f"  Rows: {pre_trim:,} → {len(combined):,}")
print(f"  Date range: {combined['date'].min().date()} → {combined['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: VALIDATE")
print("=" * 90)

# 4a. No duplicate dates
n_dupes = combined['date'].duplicated().sum()
assert n_dupes == 0, "FATAL: Duplicate dates!"
print(f"\n  ✓ No duplicate dates")

# 4b. Zero NaN in features
all_feature_cols = [c for c in combined.columns
                    if c not in ['date', 'target_daily_return', 'target_monthly_return']]
feature_nan = combined[all_feature_cols].isna().sum()
feature_nan_total = feature_nan.sum()

if feature_nan_total > 0:
    nan_cols = feature_nan[feature_nan > 0].sort_values(ascending=False)
    print(f"\n  ⚠ Feature NaN: {feature_nan_total}")
    for c in nan_cols.head(15).index:
        print(f"    {c}: {int(nan_cols[c])}")
else:
    print(f"  ✓ Zero NaN in features")

# 4c. Target checks
daily_target_nan = combined['target_daily_return'].isna().sum()
monthly_target_nan = combined['target_monthly_return'].isna().sum()
print(f"\n  Daily target NaN:   {daily_target_nan}")
print(f"  Monthly target NaN: {monthly_target_nan}")

print(f"\n  Daily target statistics:")
print(f"    Mean:  {combined['target_daily_return'].mean():.6f}")
print(f"    Std:   {combined['target_daily_return'].std():.6f}")

print(f"\n  Monthly target statistics (merge_asof to daily):")
print(f"    Mean:  {combined['target_monthly_return'].mean():.6f}")
print(f"    Std:   {combined['target_monthly_return'].std():.6f}")

# 4d. Forward-fill verification
mid_month = combined[combined['date'].dt.day.between(10, 20)].iloc[0]
sample_date_val = mid_month['date']
prev_month_end = monthly[monthly['date'] <= sample_date_val]['date'].max()

print(f"\n  Merge_asof verification:")
print(f"    Sample date: {sample_date_val.date()}")
print(f"    Most recent month-end: {prev_month_end.date() if pd.notna(prev_month_end) else 'N/A'}")

# 4e. Column breakdown
daily_feature_cols = [c for c in combined.columns
                      if not c.startswith('monthly_')
                      and c not in ['date', 'target_daily_return', 'target_monthly_return']]
monthly_feature_cols = [c for c in combined.columns if c.startswith('monthly_')]

print(f"\n  Column breakdown:")
print(f"    Daily features:         {len(daily_feature_cols)}")
print(f"    Monthly features:       {len(monthly_feature_cols)} (from merge_asof)")
print(f"    Daily target:           1")
print(f"    Monthly target:         1")
print(f"    Date:                   1")
print(f"    Total:                  {combined.shape[1]}")

# 4f. Z-score sanity (one daily factor across moments + one monthly factor across moments)
print(f"\n  Z-score sanity check:")
print(f"  {'Column':<55s} {'Mean':>8s} {'Std':>8s}")
print("  " + "-" * 75)

# Sample daily moment columns
daily_sample_base = None
for c in daily_feature_cols:
    if c.endswith('_cwmean'):
        daily_sample_base = c.replace('_cwmean', '')
        break

if daily_sample_base:
    for suffix in ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']:
        col = f'{daily_sample_base}{suffix}'
        if col in combined.columns:
            vals = combined[col].dropna()
            print(f"  {col:<55s} {vals.mean():>8.3f} {vals.std():>8.3f}")

# Sample monthly moment columns
monthly_sample_base = None
for c in monthly_feature_cols:
    if c.endswith('_cwmean'):
        monthly_sample_base = c.replace('_cwmean', '')
        break

if monthly_sample_base:
    for suffix in ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']:
        col = f'{monthly_sample_base}{suffix}'
        if col in combined.columns:
            vals = combined[col].dropna()
            print(f"  {col:<55s} {vals.mean():>8.3f} {vals.std():>8.3f}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: SAVE")
print("=" * 90)

combined = combined.sort_values('date').reset_index(drop=True)

out_path = BASE_DIR / 'model_market_combined_full_moments.parquet'
combined.to_parquet(out_path, index=False, engine='pyarrow')

file_size = out_path.stat().st_size
print(f"\n  ✓ Saved: {out_path}")
print(f"    {combined.shape[0]:,} rows × {combined.shape[1]} columns")
print(f"    Size: {file_size / 1e6:.1f} MB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("MODEL-READY COMBINED FULL MOMENTS COMPLETE")
print("=" * 90)

print(f"""
  Inputs (both already z-scored):
    Daily:   model_market_daily_full_moments.parquet   ({daily.shape[0]:,} rows × {daily.shape[1]} cols)
    Monthly: model_market_monthly_full_moments.parquet ({monthly.shape[0]} rows × {monthly.shape[1]} cols)

  Merge strategy:
    merge_asof(direction='backward') on date
    Monthly features prefixed with 'monthly_' for clarity

  Result:
    Rows:            {combined.shape[0]:,} trading days
    Columns:         {combined.shape[1]}
      Daily features:   {len(daily_feature_cols)}
      Monthly features: {len(monthly_feature_cols)}
      Targets:          2 (daily + monthly)
      Date:             1
    Dates:           {combined['date'].min().date()} → {combined['date'].max().date()}
    NaN:             {feature_nan_total} features

  ═══════════════════════════════════════════════════════════════════════════
  STAGE 3 COMPLETE — ALL 6 MODEL-READY TABLES BUILT
  ═══════════════════════════════════════════════════════════════════════════

  means_only/
    01  model_market_daily_means.parquet            4,299 rows × 400 cols
    02  model_market_monthly_means.parquet            205 rows × 326 cols
    03  model_market_combined_means.parquet          4,299 rows × 725 cols

  full_moments/
    01  model_market_daily_full_moments.parquet      4,299 rows × 1,147 cols
    02  model_market_monthly_full_moments.parquet      205 rows × 1,069 cols
    03  model_market_combined_full_moments.parquet   {combined.shape[0]:,} rows × {combined.shape[1]} cols

  All tables: zero NaN, verified targets, expanding-window z-scored,
  no look-ahead bias, ready for model development.
""")

STEP 1: LOAD BOTH Z-SCORED TABLES

  Daily:   4,299 rows × 1147 columns
           2007-11-30 → 2024-12-30
  Monthly: 205 rows × 1069 columns
           2007-11-30 → 2024-11-30

STEP 2: PREFIX MONTHLY COLUMNS

  No column name conflicts

  Prefixed 1067 monthly feature columns with 'monthly_'
  Monthly columns after rename: 1069

STEP 3: MERGE_ASOF MONTHLY TO DAILY FREQUENCY

  After merge_asof: 4,299 rows × 2215 columns
  Trimmed 0 rows before first monthly observation
  Rows: 4,299 → 4,299
  Date range: 2007-11-30 → 2024-12-30

STEP 4: VALIDATE

  ✓ No duplicate dates
  ✓ Zero NaN in features

  Daily target NaN:   0
  Monthly target NaN: 0

  Daily target statistics:
    Mean:  0.000499
    Std:   0.012568

  Monthly target statistics (merge_asof to daily):
    Mean:  0.009922
    Std:   0.045127

  Merge_asof verification:
    Sample date: 2007-12-10
    Most recent month-end: 2007-11-30

  Column breakdown:
    Daily features:         1145
    Monthly features:       1067 (from me